# 하이브리드(GCN 임베딩 + XGBoost) — Colab
같은 Colab 환경에서 **세 가지를 한꺼번에** 돌려서 보고서에 쓸 숫자를 정합니다 (환경이 다르면 점수가 조금씩 달라서 한 환경에서 같이 돌려야 비교가 공정합니다).

| 이름 | 입력 | 의미 |
|---|---|---|
| `xgb_check` | 원본 165 | 기준선 재현 (공식 `xgb`와 거의 같아야 함) |
| `hybrid_gcn_naive` | 원본 + 저장된 GCN 임베딩 | 누수 있음 |
| `hybrid_gcn` | 원본 + **OOF** GCN 임베딩 | 누수 처리 (본 실험) |

**미리 할 일:** 이 노트북이 쓰는 코드(`hybrid/run_hybrid.py`, `hybrid/make_oof_emb.py`)가 GitHub `main`에 push되어 있어야 합니다.
**런타임:** GPU(T4/L4) 권장 (OOF 임베딩 만들 때 GNN을 25번 학습). 예상 시간: 설치 포함 10~20분.

In [ ]:
# 1) 환경 확인 (결과를 README에 적을 때 같이 기록)
!python --version
!nvidia-smi -L || echo 'GPU 없음 -> 런타임 유형을 T4 GPU로 바꾸세요'
import torch; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
# 2) 저장소 가져오기: main만 받고(작게), 튜닝 후 GCN 결과가 있는 gcn 브랜치를 따로 가져옵니다
import getpass, os
if not os.path.exists('2026-2-HAI-ProjectTeam8'):
    tok = getpass.getpass('GitHub 토큰 (public이면 그냥 엔터): ')
    host = f'{tok}@' if tok else ''
    !git clone -q --single-branch -b main https://{host}github.com/HanyangTechAI/2026-2-HAI-ProjectTeam8.git
%cd 2026-2-HAI-ProjectTeam8
!git pull -q origin main || true
!git fetch -q origin gcn:refs/remotes/origin/gcn
!ls hybrid
!git log --oneline -3

위 `ls hybrid`에 `run_hybrid.py`와 `make_oof_emb.py`가 **둘 다** 보여야 합니다. 없으면 로컬에서 push가 안 된 것입니다 (`git status`로 확인 후 push하고 이 셀을 다시 실행).

In [ ]:
# 3) 데이터 복사 (드라이브 -> data/elliptic)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p data/elliptic
!cp "/content/drive/MyDrive/Colab Notebooks/elliptic/"*.csv data/elliptic/
!ls -la data/elliptic

In [ ]:
# 4) 설치: PyG + 공식 베이스라인과 같은 xgboost 버전
!pip install -q torch-geometric "xgboost>=3.4.1,<3.5" pytest scipy
import xgboost, torch_geometric
print('xgboost', xgboost.__version__, '(공식 베이스라인: Colab 3.4.1)', '| torch_geometric', torch_geometric.__version__)

In [ ]:
# 5) 튜닝 후 GCN 결과 가져오기 (gcn 브랜치): 기준 임베딩 5개 + 시드별 best_epoch가 들어 있는 metrics
#    gcn 브랜치가 main에 합쳐지고 임베딩을 드라이브로 옮기면 이 셀은 바꿔야 합니다.
!mkdir -p results/emb results_hybrid/metrics
for s in range(5):
    !git show origin/gcn:results/emb/emb_gcn_seed{s}.npy > results/emb/emb_gcn_seed{s}.npy
!git show origin/gcn:results/metrics/gcn.csv > results_hybrid/metrics/gcn.csv
import numpy as np, pandas as pd
for s in range(5):
    e = np.load(f'results/emb/emb_gcn_seed{s}.npy', mmap_mode='r'); print(s, e.shape, e.dtype)
m = pd.read_csv('results_hybrid/metrics/gcn.csv', dtype={'threshold_type': str})
t = m[(m.split == 'test') & (m.threshold_type == 'fixed_0.5')].sort_values('seed')
print(t[['seed', 'pr_auc', 'notes']].to_string(index=False))
print('튜닝 후 GCN이 맞다면 PR-AUC 평균 0.662, best_epoch 144/167/203/306/224 여야 합니다:', round(t.pr_auc.mean(), 4))

In [ ]:
# 6) 코드 점검 (가짜 데이터로 몇 십 초). 전부 passed여야 다음으로 진행
!python -m pytest -q -p no:cacheprovider tests/test_hybrid.py tests/test_oof.py

In [ ]:
# 7) 기준선 재현: 원본 165개만으로 같은 파이프라인을 돌립니다 -> xgb_check
#    공식 xgb(Colab, results/metrics/xgb.csv)와 PR-AUC 차이가 0.001 안쪽이어야 정상입니다
!python -m hybrid.run_hybrid --mode raw --results-dir results_hybrid

In [ ]:
# 8) 누수 있는 하이브리드: 저장된 GCN 임베딩 그대로 -> hybrid_gcn_naive
!python -m hybrid.run_hybrid --emb-models gcn --results-dir results_hybrid

In [ ]:
# 9) OOF 임베딩 만들기 (GPU): 시간 블록 5개 x 시드 5개 = 학습 25번, 시드별 에폭은 gcn.csv의 best_epoch
!python -m hybrid.make_oof_emb --model gcn --lr 0.01 --dropout 0.5 --weight-decay 5e-4 \
    --metrics-csv results_hybrid/metrics/gcn.csv --out-dir results_hybrid/emb_oof

In [ ]:
# 10) OOF 하이브리드 -> hybrid_gcn
!python -m hybrid.run_hybrid --emb-models gcn --emb-kind oof --emb-dir results_hybrid/emb_oof --results-dir results_hybrid

In [ ]:
# 11) 세 가지 비교 (같은 시드끼리 짝지어서)
import numpy as np, pandas as pd
from scipy import stats
from sklearn.metrics import average_precision_score
from common.data import load_data
pd.set_option('display.width', 200)
R = 'results_hybrid/'

def g(name, split='test', tt='fixed_0.5', path=None):
    m = pd.read_csv(path or f'{R}metrics/{name}.csv', dtype={'threshold_type': str})
    return m[(m.split == split) & (m.threshold_type == tt)].sort_values('seed').set_index('seed')

off, raw, nv, oo = g('xgb', path='results/metrics/xgb.csv'), g('xgb_check'), g('hybrid_gcn_naive'), g('hybrid_gcn')
tab = pd.DataFrame({'xgb(공식)': off.pr_auc, 'xgb_check': raw.pr_auc, '누수 있음': nv.pr_auc, 'OOF': oo.pr_auc})
tab['OOF-xgb_check'] = tab['OOF'] - tab['xgb_check']
print('== 테스트 PR-AUC (시드별) ==')
print(tab.round(4).to_string())
print('\n평균 ± 표준편차:', {c: f'{tab[c].mean():.4f} ± {tab[c].std(ddof=1):.4f}' for c in tab.columns})
d = tab['OOF-xgb_check'].values
print(f'\nOOF - xgb_check: {d.mean():+.4f} ± {d.std(ddof=1):.4f} | 시드 {int((d > 0).sum())}/5에서 OOF가 높음 | 대응 t검정 p={stats.ttest_rel(oo.pr_auc, raw.pr_auc).pvalue:.3f}')

print('\n== F1 / 정밀도 / 재현율 (테스트) ==')
for tt in ('fixed_0.5', 'val_opt'):
    rows = [(l, g(n, 'test', tt).f1.mean(), g(n, 'test', tt).f1.std(ddof=1), g(n, 'test', tt).precision.mean(), g(n, 'test', tt).recall.mean())
            for n, l in (('xgb_check', 'xgb_check'), ('hybrid_gcn_naive', '누수 있음'), ('hybrid_gcn', 'OOF'))]
    print(tt); print(pd.DataFrame(rows, columns=['모델', 'F1', 'F1 std', '정밀도', '재현율']).round(4).to_string(index=False))
    dd = (g('hybrid_gcn', 'test', tt).f1 - g('xgb_check', 'test', tt).f1).values
    print(f'  F1 차이(OOF - xgb_check): {dd.mean():+.4f} ± {dd.std(ddof=1):.4f} (시드 {int((dd > 0).sum())}/5 높음)')

print('\n== 검증 PR-AUC (①모델) ==  누수 있으면 부풀어 있어야 함')
print({l: round(g(n, 'val', 'val_opt').pr_auc.mean(), 4) for n, l in (('xgb_check', 'xgb_check'), ('hybrid_gcn_naive', '누수 있음'), ('hybrid_gcn', 'OOF'))})

data = load_data('data/elliptic', standardize=False)
y, ts, te = data.y, data.time_step, data.test_mask
print('\n== 테스트 구간별 PR-AUC (시드 평균 ± 표준편차) ==')
for sname, m in {'35~42': te & (ts <= 42), '43~49': te & (ts >= 43)}.items():
    out = []
    for n, l in (('xgb_check', 'xgb_check'), ('hybrid_gcn_naive', '누수 있음'), ('hybrid_gcn', 'OOF')):
        v = [average_precision_score(y[m], np.load(f'{R}pred/pred_{n}_seed{s}.npy')[m]) for s in range(5)]
        out.append(f'{l} {np.mean(v):.4f}±{np.std(v, ddof=1):.4f}')
    print(sname, ' | '.join(out), f'| 불법 {int((y[m] == 1).sum())}건')

imp = pd.read_csv(f'{R}hybrid_importance_hybrid_gcn.csv')
print('\nOOF gain 비중(시드 평균):', imp.groupby('group').gain_share.mean().round(3).to_dict())
print('\n== OOF 폴드 진단 (시드 평균) ==')
rep = pd.read_csv(f'{R}emb_oof/oof_report_gcn.csv')
print(rep.groupby('fold')[['cos_raw', 'cos_aligned_unlabeled', 'cos_aligned_labeled', 'oof_pr_auc']].mean().round(3).to_string())

In [ ]:
# 12) 결과 백업(드라이브) + 작은 파일만 zip으로 다운로드 (emb_oof는 용량이 커서 드라이브에만 보관)
import os, shutil
from google.colab import files
dst = '/content/drive/MyDrive/Colab Notebooks/elliptic/hybrid_results'
os.makedirs(dst, exist_ok=True)
!cp -r results_hybrid/. "{dst}"
!rm -rf out_small && mkdir -p out_small/emb_oof && cp -r results_hybrid/metrics results_hybrid/pred out_small/ && cp results_hybrid/*.csv out_small/ && cp results_hybrid/emb_oof/oof_report_gcn.csv out_small/emb_oof/
shutil.make_archive('hybrid_results', 'zip', 'out_small')
print('드라이브:', dst); files.download('hybrid_results.zip')